# 💻 Chapter 14: Monte Carlo Methods
*Track 2 — Developers | Tier 2*

> **🎬 Hook:** Estimate π by throwing darts. Then use the same idea to price financial derivatives, validate APIs, and solve integrals that don't have closed forms.

**🎯 Objectives:** Build Monte Carlo estimators, understand convergence, apply to integration and simulation.

## 📖 Section 1 — Concept Review

### The Monte Carlo Principle
Use random sampling to estimate quantities that are:
- Too complex to compute analytically
- High-dimensional integrals
- System behavior under uncertainty

### The Core Idea
If X is a random variable and we want E[g(X)]:
$$E[g(X)] \approx \frac{1}{n} \sum_{i=1}^n g(x_i) \quad \text{where } x_i \sim p(x)$$

### Convergence Rate
Error ∝ 1/√n (regardless of dimension!)
This is Monte Carlo's killer advantage in high dimensions.

### Variance Reduction Techniques
1. **Antithetic variates**: Use both x and (1-x) for U[0,1]
2. **Control variates**: Subtract a correlated variable with known mean
3. **Importance sampling**: Sample from regions that matter most
4. **Stratified sampling**: Divide the domain, sample from each region

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, integrate
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

# ── Monte Carlo Integration ──
def mc_integrate(f, a, b, n, seed=42):
    rng = np.random.default_rng(seed)
    x = rng.uniform(a, b, n)
    return (b - a) * f(x).mean()

# Integrate sin(x) from 0 to π = 2
f = np.sin
a, b = 0, np.pi
true_val = 2.0

ns = np.logspace(1, 6, 50).astype(int)
estimates = [mc_integrate(f, a, b, n) for n in ns]
errors = [abs(e - true_val) for e in estimates]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Show the integral
x_plot = np.linspace(0, np.pi, 300)
axes[0].plot(x_plot, np.sin(x_plot), lw=3, color='#3498db', label='sin(x)')
axes[0].fill_between(x_plot, np.sin(x_plot), alpha=0.3, color='#3498db', label='Integral = 2.0')
mc_pts_x = rng.uniform(0, np.pi, 200)
mc_pts_y = rng.uniform(0, 1, 200)
hit = mc_pts_y <= np.sin(mc_pts_x)
axes[0].scatter(mc_pts_x[hit], mc_pts_y[hit], s=10, c='#27ae60', alpha=0.7, label='Inside')
axes[0].scatter(mc_pts_x[~hit], mc_pts_y[~hit], s=10, c='#e74c3c', alpha=0.7, label='Outside')
axes[0].set_title(f'∫sin(x)dx from 0 to π = 2.000
MC estimate: {mc_integrate(f,a,b,10000):.4f}', fontweight='bold')
axes[0].legend(fontsize=8)

# Error convergence
axes[1].loglog(ns, errors, 'bo-', markersize=4, lw=1.5, label='MC error')
axes[1].loglog(ns, 1/np.sqrt(ns), 'r--', lw=2, label='O(1/√n) reference')
axes[1].set_xlabel('n (samples)'); axes[1].set_ylabel('Absolute Error')
axes[1].set_title('Monte Carlo Convergence: O(1/√n)', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('ch14_mc_integration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Variance Reduction: Antithetic Variates ──
rng = np.random.default_rng(42)

def estimate_normal_prob(n_samples):
    # Estimate P(Z > 2) for Z~N(0,1) using different methods.
    # Standard MC
    z_plain = rng.normal(0, 1, n_samples)
    plain = (z_plain > 2).mean()

    # Antithetic variates: use z and -z together
    z_half = rng.normal(0, 1, n_samples // 2)
    z_anti = np.concatenate([z_half, -z_half])
    antithetic = (z_anti > 2).mean()

    true_val = 1 - stats.norm.cdf(2)
    return plain, antithetic, true_val

results = []
for n in [100, 1000, 10000, 100000]:
    p, a, t = estimate_normal_prob(n)
    results.append({'n': n, 'plain_err': abs(p-t), 'antithetic_err': abs(a-t)})
    print(f"n={n:>7}: Plain error={abs(p-t):.5f}, Antithetic error={abs(a-t):.5f}")

print(f"
True P(Z>2) = {1-stats.norm.cdf(2):.6f}")
print("Antithetic variates often give 2-4x variance reduction!")

In [ ]:
# ── Applied: Option Pricing (Black-Scholes via Monte Carlo) ──
rng = np.random.default_rng(42)

# European call option parameters
S0 = 100    # current stock price
K  = 105    # strike price
T  = 1.0    # time to expiry (years)
r  = 0.05   # risk-free rate
sigma = 0.2 # volatility

def mc_option_price(S0, K, T, r, sigma, n_paths=100_000):
    rng = np.random.default_rng(42)
    # Geometric Brownian Motion: S_T = S0 * exp((r - σ²/2)T + σ√T·Z)
    Z = rng.normal(0, 1, n_paths)
    ST = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    payoff = np.maximum(ST - K, 0)
    price = np.exp(-r*T) * payoff.mean()
    se = np.exp(-r*T) * payoff.std() / np.sqrt(n_paths)
    return price, se, ST

price, se, ST = mc_option_price(S0, K, T, r, sigma)

# Black-Scholes analytical formula for comparison
d1 = (np.log(S0/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)
bs_price = S0*stats.norm.cdf(d1) - K*np.exp(-r*T)*stats.norm.cdf(d2)

print("📈 European Call Option Pricing")
print(f"  MC Price: ${price:.4f} ± {1.96*se:.4f} (95% CI)")
print(f"  Black-Scholes: ${bs_price:.4f}")
print(f"  Error: ${abs(price-bs_price):.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(ST, bins=60, color='#3498db', edgecolor='white', density=True, alpha=0.7)
ax1.axvline(K, color='red', lw=2, linestyle='--', label=f'Strike K={K}')
ax1.set_title('Distribution of Stock Price at T=1yr', fontweight='bold')
ax1.set_xlabel('Stock Price'); ax1.legend()

payoffs = np.maximum(ST - K, 0)
ax2.hist(payoffs[payoffs > 0], bins=50, color='#27ae60', edgecolor='white', density=True, alpha=0.7)
ax2.set_title(f'Positive Payoffs (options in-the-money)
Mean payoff = ${np.exp(-r*T)*np.maximum(ST-K,0).mean():.2f}', fontweight='bold')
ax2.set_xlabel('Payoff ($)')
plt.tight_layout()
plt.savefig('ch14_option_pricing.png', dpi=150, bbox_inches='tight')
plt.show()

## ✏️ Section 6 — Coding Challenges

**Challenge 1:** Use Monte Carlo to estimate ∫₀¹ e^(-x²) dx.
Compare with `scipy.integrate.quad`.

**Challenge 2:** Simulate a system: 3 independent components, each fails with P=0.1.
System fails if ANY component fails. Estimate P(system failure) via Monte Carlo.
Verify analytically.

**Challenge 3:** Implement importance sampling to estimate P(Z > 4) more efficiently than plain MC.

<details><summary>Solutions</summary>

**C1:** True = 0.7468. MC with n=100k gets within 0.001.

**C2:** P(system failure) = 1 - (0.9)³ = 0.271. MC should give ~0.271.

**C3:** Sample from N(4, 1) instead of N(0,1), then reweight by likelihood ratio.
</details>

In [ ]:
# C1: MC Integration vs scipy
from scipy.integrate import quad

f = lambda x: np.exp(-x**2)
mc_est = mc_integrate(np.vectorize(f), 0, 1, 1_000_000)
scipy_val, _ = quad(f, 0, 1)
print(f"MC estimate: {mc_est:.6f}")
print(f"Scipy exact: {scipy_val:.6f}")
print(f"Error: {abs(mc_est-scipy_val):.6f}")

# C2: System reliability
rng = np.random.default_rng(42)
n = 1_000_000
comps = rng.random((n, 3)) < 0.1  # each component fails with P=0.1
system_fails = comps.any(axis=1)
print(f"
System failure (simulated): {system_fails.mean():.4f}")
print(f"System failure (theory):   {1 - 0.9**3:.4f}")

## 🎯 Recap
1. Monte Carlo = estimate by averaging over random samples.
2. Convergence is O(1/√n) — always, regardless of dimension.
3. Variance reduction (antithetic, stratification) can give big speedups.

**Next:** [Chapter 15 — A/B Testing]